# Testing the Medical Analysis RAG Chatbot
This Jupyter Notebook is designed for testing the core logic and functionalities of our Medical Analysis RAG Chatbot. Here, we will systematically evaluate different components, including document retrieval and the chatbot's response generation capabilities.


In [16]:
import os
import json 
from rouge_score import rouge_scorer
from backend.app.modules.document_preprocessing import PDFExtraction, CleanText, split_text
from backend.app.modules.document_retrieval import store_embeddings, retrieve_text
from backend.app.modules.chatbot_logic import Chatbot 
from backend.app.config import (
    OPENAI_KEY,
    MINI_LM_EMBED,
    OPENAI_EMBED,
    VECTOR_DB_PATH,
    PROJECT_ROOT
)
DEFAULT_EMBED_MODEL = OPENAI_EMBED

In [5]:
question_answer_pairs = [
  {
    "question": "Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?",
    "answer_reference": "Empfehlung: MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund."
  },
  {
    "question": "Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?",
    "answer_reference": "Empfehlung: MRT beider Hüften bei einseitiger Femurkopfnekrose im ARCO Stadium I-IV."
  },
  {
    "question": "Welche Klassifikation wird zur Stadieneinteilung der atraumatischen Femurkopfnekrose empfohlen?",
    "answer_reference": "Empfehlung: Nutzung der modifizierten ARCO-Klassifikation."
  },
  {
    "question": "Wie sollte bei Verdacht auf eine subchondrale Fraktur im ARCO Stadium II, aber unklarer Diagnostik, weiter vorgegangen werden?",
    "answer_reference": "Empfehlung: Durchführung einer CT zur Klärung der subchondralen Fraktur."
  },
  {
    "question": "Sollte eine Szintigraphie zur Diagnostik der atraumatischen Femurkopfnekrose eingesetzt werden?",
    "answer_reference": "Empfehlung: Szintigraphie wird nicht zur Diagnostik der atraumatischen Femurkopfnekrose empfohlen."
  },
  {
    "question": "Wie differenziert man im MRT zwischen einem transitorischen Knochenmarködem und einer Osteonekrose?",
    "answer_reference": "Empfehlung: MRT-Muster und klinischer Verlauf sind entscheidend für die Differenzierung."
  },
  {
    "question": "Welche bildgebende Methode gilt als Goldstandard in der Diagnostik der atraumatischen Femurkopfnekrose?",
    "answer_reference": "Empfehlung: MRT als Goldstandard."
  },
  {
    "question": "Welche Bildgebung eignet sich am besten zur Detektion einer subchondralen Fraktur?",
    "answer_reference": "Empfehlung: CT für die Darstellung subchondraler Frakturen."
  },
  {
    "question": "Welche Risikofaktoren sprechen für eine bilaterale Beteiligung bei Femurkopfnekrose?",
    "answer_reference": "Empfehlung: Einseitige Femurkopfnekrose erhöht das Risiko für bilaterale Erkrankung; Risikofaktoren beachten."
  },
  {
    "question": "Welche röntgenologischen Befunde charakterisieren das ARCO Stadium III?",
    "answer_reference": "Empfehlung: Zeichen der subchondralen Fraktur mit beginnender Gelenkflächeninkongruenz im Röntgenbild."
  }
]

qa_dictionary = {}

for item in question_answer_pairs:
    question = item["question"]
    answer = item["answer_reference"]
    qa_dictionary[question] = answer

print(qa_dictionary)

{'Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?': 'Empfehlung: MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund.', 'Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?': 'Empfehlung: MRT beider Hüften bei einseitiger Femurkopfnekrose im ARCO Stadium I-IV.', 'Welche Klassifikation wird zur Stadieneinteilung der atraumatischen Femurkopfnekrose empfohlen?': 'Empfehlung: Nutzung der modifizierten ARCO-Klassifikation.', 'Wie sollte bei Verdacht auf eine subchondrale Fraktur im ARCO Stadium II, aber unklarer Diagnostik, weiter vorgegangen werden?': 'Empfehlung: Durchführung einer CT zur Klärung der subchondralen Fraktur.', 'Sollte eine Szintigraphie zur Diagnostik der atraumatischen Femurkopfnekrose eingesetzt werden?': 'Empfehlung: Szintigraphie wird nicht zur Diagnostik der atraumatischen Femurkopfnekrose empfohlen.', 'Wie differenziert man im MRT zwischen einem transitorischen K

#### 2. Document Preprocessing

In the initial stage of our pipeline, we performed document preprocessing on the provided German medical guideline (`Guideline_atraumatische_Femurkopfnekrose_2019-09_1-abgelaufen.pdf`). This step, implemented in the `document_preprocessing` module, aims to clean and prepare the raw text extracted from the PDF for further processing.

The `CleanText` class within this module applies a series of transformations to the text. For instance:

* **Header and Footer Removal:** Standard headers and footers, such as "S3-Leitlinie Atraumatische Femurkopfnekrose..." and page numbers, were removed to focus on the core content.
* **Reference Preservation:** Citation markers like `[12]` are preserved and transformed into `REF12` to maintain the integrity of medical references without interfering with text splitting.
* **Hyphenation Handling:** Broken compound words at the end of lines (e.g., "Core -\ndecompression") were corrected to "Core decompression". Spacing around hyphens was also normalized.
* **Special Character Cleaning:** Non-alphanumeric characters (excluding those commonly found in medical text like periods, commas, semicolons, colons, hyphens, parentheses, slashes, and degree symbols) were removed to reduce noise.
* **Whitespace Normalization:** Multiple spaces and inconsistent line breaks were standardized to improve text consistency. Paragraph breaks were carefully preserved.

#### 3. Text Splitting

Following the cleaning process, the preprocessed text was split into smaller, manageable chunks using the `split_text` function from the `document_preprocessing` module. This function utilizes the `RecursiveCharacterTextSplitter` from Langchain, configured with a `chunk_size` of 1000 characters and a `chunk_overlap` of 200 characters.

Given the substantial length of the "Guideline_atraumatische_Femurkopfnekrose..." PDF, even with a `chunk_size` of 1000 characters, the document was segmented into over 470 individual chunks.

The decision to use a `chunk_size` of 1000 was a deliberate trade-off. While a smaller `chunk_size` would result in more granular pieces of information, potentially fitting more easily within the context window limitations of language models, it could also lead to a loss of broader contextual understanding. Conversely, significantly increasing the `chunk_size`, while preserving more context within each chunk, risks exceeding the input limits of the model and potentially diluting the relevance of specific information within a large chunk.

An `overlap` of 200 characters was chosen to ensure that there is sufficient contextual continuity between consecutive chunks. This overlap helps the language model maintain a better understanding of the relationships between different parts of the document when processing the retrieved information.

The resulting text chunks, each containing a segment of the cleaned medical guideline, are then used in the subsequent steps of the RAG pipeline for embedding and retrieval. The extracted chunks were saved to `text_chunks.json` for verification.

In [ ]:
"""
In order to later run the queries, you would need a vector database. Please run the following script. 
The script will create a vector database with the name 'medical_dataset' and store the embeddings of the text chunks 
in it.
"""

pdf_path = os.path.join(os.getcwd(), "backend", "app", "documents", "Guideline_atraumatische_Femurkopfnekrose_2019-09_1-abgelaufen.pdf")
    
pdf_extractor = PDFExtraction()
raw_text = pdf_extractor.extract_text_from_pdf(pdf_path)

if isinstance(raw_text, list):
    raw_text = "\n".join(raw_text)
    
cleaner = CleanText(raw_text)
cleaned_text = cleaner.clean()

text_chunks = split_text([cleaned_text], chunk_size=1000, chunk_overlap=200)
print(f"Extracted {len(text_chunks)} text chunks.")

store_embeddings(text_chunks, embed_model=OPENAI_EMBED, collection_name="medical_dataset")

Extracted 481 text chunks.


In [14]:
"""
To evaluate the accuracy of our retrieval mechanism, this notebook performs the following:

1.  Retrieves the top three most relevant passages for given queries.
2.  Calculates accuracy scores for these retrieved passages.
3.  Determines the precision and recall of the retrieval process.

The evaluation results are saved to a JSON file within the `Extra/` directory.
"""
results = []
vectordb_path = os.path.join(os.getcwd(), "backend", "app", "vector_store", "medical_dataset")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for question, reference_answer in qa_dictionary.items():
        matched_texts = retrieve_text(
            vectordb_path=vectordb_path,
            query=question,
            embed_model=OPENAI_EMBED,
            collection_name="medical_dataset",
            results_to_return=3
        )

        retrieved_info = []
        for i, (doc, chroma_score) in enumerate(matched_texts):
            text = doc.page_content
            scores = scorer.score(reference_answer, text)
            retrieved_info.append({
                "passage": text,
                "chroma_score": chroma_score,
                "rouge_scores": {
                    metric: {
                        "precision": scores[metric].precision,
                        "recall": scores[metric].recall,
                        "f1": scores[metric].fmeasure
                    }
                    for metric in ['rouge1', 'rouge2', 'rougeL']
                }
            })

        results.append({
            "question": question,
            "reference_answer": reference_answer,
            "retrieved_information": retrieved_info
        })

output_file_path = "Extra/retrieval_evaluation_results.json"  # Specify the desired path
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Results saved to: {output_file_path}")




Retrieving from Vector DB Path: /Users/syedalimuradtahir/Documents/Personal_Projects/RAG-Chatbot/backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Retrieving from Vector DB Path: /Users/syedalimuradtahir/Documents/Personal_Projects/RAG-Chatbot/backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?
Retrieving from Vector DB Path: /Users/syedalimuradtahir/Documents/Personal_Projects/RAG-Chatbot/backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Welche Klassifikation wird zur Stadieneinteilung der atraumatischen Femurkopfnekrose empfohlen?
Retrieving from Vector DB Path: /Users/syedalimuradtahir/Documents/Personal_Projects/RAG-Chatbot/backend/app/vector_store/medical_data

The low ROUGE precision scores, particularly for Rouge-1 and Rouge-L, suggest that while the retrieved passage contains many of the words present in the concise reference answer (as indicated by the higher recall), it also includes a significant amount of additional, potentially irrelevant information. This is likely due to our larger chunk size during document splitting. While a larger chunk size provides more surrounding context, which can be beneficial for the chatbot's comprehension later, it can negatively impact precision-focused metrics like ROUGE when comparing against a short, precise reference answer. The ROUGE score emphasizes exact word overlap, and the extra context in the larger chunks introduces words not present in the brief reference.

The very low Rouge-2 scores further highlight the lack of direct phrasal overlap. This indicates that the retrieved information, even if it contains the right keywords, might not be phrased similarly to the expected answer.

Therefore, while these retrieval results might offer sufficient context for the downstream chatbot model to synthesize a good answer, they don't achieve a high degree of word-for-word overlap with the concise reference answer, leading to lower precision scores. This trade-off between broader context and precise keyword matching is an important consideration in RAG system design.

In [17]:
"""
Now we will evaluate the chatbot's performance using the same questions and reference answers. 
Since chatbot model is mainly used for generating answers by extracting information from the vector database,
we will use the same questions and reference answers as before.
"""


chatbot = Chatbot(model_type='openai')
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
evaluation_results = []

for item in question_answer_pairs:
    question = item["question"]
    reference_answer = item["answer_reference"]
    chatbot.clear_history()  

    generated_answer = chatbot.generate_text(question)
    scores = scorer.score(reference_answer, generated_answer)

    evaluation_results.append({
        "question": question,
        "reference_answer": reference_answer,
        "generated_answer": generated_answer,
        "rouge_scores": {
            metric: {
                "precision": scores[metric].precision,
                "recall": scores[metric].recall,
                "f1": scores[metric].fmeasure
            }
            for metric in ['rouge1', 'rouge2', 'rougeL']
        }
    })

output_file_path = "Extra/chatbot_evaluation_results.json"
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=4)

print(f"Chatbot evaluation results saved to: {output_file_path}")

Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?
Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Welche Klassifikation wird zur Stadieneinteilung der atraumatischen Femurkopfnekrose empfohlen?
Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset
Number of Retrieved Texts: 3
Query used for retrieval: Wie sollte bei Verdacht auf eine subchondrale Fraktur im ARCO Stadium II, aber unklarer Diagnostik, weiter vorgegangen werden?
Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset
N

##### Final Takeaway: Don’t Let the Scores Fool You  

**The model’s answers are clinically correct** and align with the reference recommendations, *even though ROUGE scores look low*. Here’s why the scores aren’t telling the full story:  

1. **ROUGE measures wording, not meaning**.  
   - It’s like judging a recipe by how many times the word “salt” appears, rather than whether the dish tastes right. The model paraphrases answers (e.g., “MRT des Hüftgelenkes” vs. “MRT”) or adds clarifications (e.g., explaining *why* CT is better for fractures), which tanks precision scores but doesn’t make the answer wrong.  

2. **Medical accuracy ≠ word-for-word copying**.  
   - If the answer correctly recommends **MRI for persistent pain** or **CT for subchondral fractures**, it’s clinically valid even if ROUGE penalizes it for not mirroring the reference verbatim.  

3. **ROUGE ignores context and expertise**.  
   - A bullet-point explanation of MRI findings (model) vs. a vague “MRT-Muster” reference? The model’s answer might actually be *more helpful* in practice, but ROUGE sees it as a mismatch.  

**Bottom line**: ROUGE is a blunt tool for medical QA. Trust human review or clinical validation over these scores they miss the nuance of *correctness* in real-world scenarios.  